# Робот, камера и карта — преподавательская версия

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elena-kornaeva/AGLA/blob/main/weeks/week-06-coordinate-systems/notebooks/robot-camera-map-teacher.ipynb)

Практика к лекции № 4 · редакционная неделя 06 · 35–45 минут + 15–25 минут исследования.

**Вопрос:** почему правильные числа датчика могут привести робота не туда?
После работы вы сможете переводить координаты точки и перемещения, отличать движение от смены описания и измерять длину в косоугольном базисе.

Это синтетическая 2D-комната с уже найденными маяками, а не перспективная камера. Внешних данных и изображений нет. Источник замысла: [сценарий занятия](../../../docs/notebook-scenarios.md#1-робот-камера-и-карта); [соглашения](../../../docs/notation.md). Исходная презентация недоступна: её дополнительные детали не воспроизводятся.

Запускайте ячейки по порядку. В упражнениях заменяйте `None` своим решением и повторно запускайте проверку. Пустые упражнения не прерывают запуск. Python 3, NumPy, Matplotlib; установки, сеть и GPU не нужны.

**Как работать:** сначала запишите прогноз в Markdown-ячейке (двойной щелчок), затем выполните опыт. В ячейках «Ваш код» изменяйте только указанные строки. После каждого изменения запускайте сначала эту ячейку, затем следующую самопроверку.

Сообщения **НЕ ЗАПОЛНЕНО**, **НУЖНО ИСПРАВИТЬ** и **УСПЕХ** обозначают состояние вашего задания. Запуск всех ячеек с пустыми ответами сам по себе не означает выполнения практики. При ошибке Python в своей ячейке исправьте указанную строку и запустите её повторно.


## Преподавателю
Это отдельная версия с заполненными решениями. Студентам выдавайте [версию с упражнениями](robot-camera-map.ipynb). Основной путь: разделы 1–4, около 37 минут; раздел 5 — дополнительно.

Просите прогноз до демонстрации и объяснение после сообщения «УСПЕХ». Численное совпадение не доказывает понимания: попросите объяснить направление действия P и повторить опыт с другим началом.

Типичные ошибки: забытый перенос; умножение на P вместо решения системы; транспонирование для любого базиса; правильные функции, но прежняя строка навигации; обычная норма в косом базисе. Сообщения проверок помогают различить эти случаи. Три проверки функций не являются доказательством их универсальной корректности — просмотрите формулу студента.

**Оценка:** основной код принят, если два обязательных задания имеют «УСПЕХ», студент объясняет перенос и заполняет таблицу раздела 4. Дополнительный уровень принят после матрицы Грама и объяснения вырождения. Отдельно оцените самостоятельность вывода, не только запуск ячеек.

## Подготовка · запустите один раз
Следующая свёрнутая ячейка загружает NumPy и Matplotlib, определяет рисование и проверки. Нажмите ▶ и дождитесь сообщения «Подготовка завершена». Читать или менять служебные функции для выполнения практики не требуется. После перезапуска ядра выполните эту ячейку снова.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("default")

def checked_basis(P):
    P = np.asarray(P, dtype=float)
    if P.shape != (2, 2) or not np.isfinite(P).all():
        raise ValueError("Нужна конечная матрица 2×2.")
    condition = np.linalg.cond(P)
    if not np.isfinite(condition) or condition >= 1e10:
        raise ValueError("Базис вырожден или почти вырожден: надёжный перевод остановлен.")
    return P

def rotation(degrees):
    t = np.deg2rad(degrees)
    return np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])

def scene(a=(2, 0), angle=90, target="A", corrected=False, skew=0., decompose=False, destination_override=None):
    # Служебный перевод для визуализации; функции студента появятся ниже.
    a = np.asarray(a, dtype=float)
    P = checked_basis(rotation(angle) @ np.array([[1., skew], [0., 1.]]))
    beacons = {"A": np.array([1., 3.]), "B": np.array([4., 3.])}
    x = beacons[target]
    y = np.linalg.solve(P, x-a)
    wrong = y.copy()
    destination = (x if corrected else wrong) if destination_override is None else np.asarray(destination_override, dtype=float)
    robot = np.array([0., 0.])
    room = np.array([[-1,-1], [6,-1], [6,5], [-1,5], [-1,-1]], float)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.8), layout="constrained")
    transforms = [lambda v: np.asarray(v), lambda v: np.linalg.solve(P, np.asarray(v)-a)]
    for k, (ax, transform) in enumerate(zip(axes, transforms)):
        outline = np.array([transform(v) for v in room])
        ax.plot(*outline.T, color="0.65", label="Стены комнаты")
        for name, point in beacons.items():
            q=transform(point)
            ax.scatter(*q, marker="*", s=180, color="#1565c0", label=f"Маяк {name}" if name==target else None)
            ax.annotate(name, q, xytext=(8, 9), textcoords="offset points")
        for point, marker, label, color in [(robot,"s","Старт робота / O карты","#222222"),
                (a,"^","Начало камеры O′","#8e24aa"), (wrong,"x","Ошибка: пакет как x","#d97706")]:
            q=transform(point)
            ax.scatter(*q, marker=marker, s=65, color=color, label=label)
        start, end=transform(robot), transform(destination)
        ax.annotate("", end, start, arrowprops=dict(arrowstyle="->", lw=2, color="#222222", linestyle="-" if corrected else "--"))
        ax.plot([], [], color="#222222", linestyle="-" if corrected else "--", label="Ваш маршрут" if destination_override is not None else ("Верный маршрут" if corrected else "Ошибочный маршрут"))
        for j in range(2):
            origin=transform(a); tip=transform(a+P[:,j])
            ax.annotate("", tip, origin, arrowprops=dict(arrowstyle="->", color="#8e24aa"))
            ax.annotate(f"g{j+1}", tip, xytext=(4,-13), textcoords="offset points", backgroundcolor="white")
        if k==0:
            for t in range(-5, 7):
                for j in range(2):
                    points=np.array([a + P @ (np.array([t,s]) if j==0 else np.array([s,t])) for s in [-7,7]])
                    ax.plot(*points.T, color="#8e24aa", alpha=.09)
        if decompose:
            nodes=[np.zeros(2), a, a+y[0]*P[:,0], x]
            for u,v,label in zip(nodes[:-1],nodes[1:],["a", "y₁g₁", "y₂g₂"]):
                u,v=transform(u),transform(v)
                ax.annotate("",v,u,arrowprops=dict(arrowstyle="->",color="#00838f",lw=2))
                ax.annotate(label,(u+v)/2,xytext=(5,5),textcoords="offset points",backgroundcolor="white")
        extent=np.vstack([outline, transform(wrong), transform(a), transform(robot), transform(destination)])
        ax.set_xlim(extent[:,0].min()-1,extent[:,0].max()+1)
        ax.set_ylim(extent[:,1].min()-1,extent[:,1].max()+1)
        ax.set_aspect("equal"); ax.grid(alpha=.2)
        ax.set_xlabel("x₁ карты" if k==0 else "y₁ камеры")
        ax.set_ylabel("x₂ карты" if k==0 else "y₂ камеры")
        ax.set_title("Карта: физическая геометрия" if k==0 else "Та же сцена: координаты камеры")
    # Один масштаб единицы для обеих панелей; в косом базисе это масштаб чисел.
    span = max(max(np.diff(ax.get_xlim())[0], np.diff(ax.get_ylim())[0]) for ax in axes)
    for ax in axes:
        cx, cy = np.mean(ax.get_xlim()), np.mean(ax.get_ylim())
        ax.set_xlim(cx-span/2, cx+span/2)
        ax.set_ylim(cy-span/2, cy+span/2)
    axes[0].legend(loc="upper left", fontsize=8)
    error=np.linalg.norm(destination-x)
    fig.suptitle(f"Маяк {target}: x={np.round(x,2)}, y={np.round(y,2)} | до цели от остановки: {error:.3f}")
    if skew:
        axes[1].text(.02,.02,"Косые координаты: длины на рисунке\nне равны физическим без метрики M",transform=axes[1].transAxes,fontsize=8,backgroundcolor="white")
    return fig


def show_skew(P_skew, y_shift):
    physical_shift = P_skew @ y_shift
    fig, ax = plt.subplots(figsize=(6,4), layout="constrained")
    for end, style, label in [(y_shift,"--","Числа y как точка: неверная геометрия"),
                             (physical_shift,"-","Физическое перемещение Py")]:
        ax.annotate("",end,(0,0),arrowprops=dict(arrowstyle="->",linestyle=style,lw=2))
        ax.plot([],[],style,color="black",label=label)
    ax.plot([0,1,2],[0,0,1],"o:",color="#1565c0",label="Шаги g₁, затем g₂")
    ax.set(xlim=(-.3,2.6),ylim=(-.3,1.8),xlabel="x₁ карты",ylabel="x₂ карты",
           title=f"Норма чисел: {np.linalg.norm(y_shift):.3f}; физическая длина: {np.linalg.norm(physical_shift):.3f}")
    ax.set_aspect("equal"); ax.grid(alpha=.2); ax.legend(fontsize=8)
    return fig


def numeric(value, shape):
    if value is None:
        return None
    try:
        array = np.asarray(value, dtype=float)
    except (TypeError, ValueError):
        return None
    return array if array.shape == shape and np.isfinite(array).all() else None


def check_coordinates(answer, origin, basis, point):
    if answer is None:
        print("НЕ ЗАПОЛНЕНО: замените None в y_attempt и запустите упражнение, затем проверку.")
        return False
    value = numeric(answer, (2,))
    if value is None:
        print("НУЖНО ИСПРАВИТЬ: ожидается массив двух конечных чисел формы (2,), например np.array([число, число]).")
        return False
    recovered = origin + basis @ value
    residual = np.linalg.norm(recovered-point)
    print("Полученная точка карты:", recovered, "| невязка:", round(residual, 6))
    ok = np.allclose(recovered, point)
    print("УСПЕХ: обратная подстановка вернула исходную точку." if ok else
          "НУЖНО ИСПРАВИТЬ: точка не совпала. Проверьте вычитание начала и направление перехода.")
    return bool(ok)


def check_navigation(to_camera, to_map, destination):
    cases = [(np.array([2.,0.]), np.array([[0.,-1.],[1.,0.]]), np.array([1.,3.]), np.array([3.,1.])),
             (np.array([1.,-1.]), np.array([[1.,1.],[0.,1.]]), np.array([3.,0.]), np.array([1.,1.])),
             (np.array([-2.,1.]), np.array([[2.,0.],[0.,3.]]), np.array([2.,-2.]), np.array([2.,-1.]))]
    passed = True
    for label, fn, reverse in [("to_camera", to_camera, False), ("to_map", to_map, True)]:
        for origin, basis, point, coords in cases:
            try:
                answer = fn(coords if reverse else point, origin, basis)
            except Exception as error:
                print(f"НУЖНО ИСПРАВИТЬ {label}: {type(error).__name__}: {error}")
                passed = False
                break
            value = numeric(answer, (2,))
            if value is None:
                print(f"{label}: НЕ ЗАПОЛНЕНО или неверный формат. Верните массив двух конечных чисел (2,).")
                passed = False
                break
            expected = point if reverse else coords
            if not np.allclose(value, expected):
                print(f"НУЖНО ИСПРАВИТЬ {label}: при a={origin}, P={basis.tolist()} получено {value}.")
                print("Проверьте перенос начала и направление перехода; транспонирование работает не для любого базиса.")
                passed = False
                break
        else:
            print(f"УСПЕХ: {label} прошла три разных примера.")
    goal = np.array([1.,3.])
    value = numeric(destination, (2,))
    if value is None:
        print("Навигация: НЕ ЗАПОЛНЕНО или неверный формат destination_attempt.")
        return False
    error = np.linalg.norm(value-goal)
    print(f"Ваша остановка: {value}; промах: {error:.6f}.")
    if not np.allclose(value, goal):
        print("НУЖНО ИСПРАВИТЬ destination_attempt: вызовите свою функцию перевода пакета на карту.")
        passed = False
    scene(corrected=False)
    scene(corrected=True, destination_override=value)
    plt.show()
    print("УСПЕХ: функции и маршрут верны." if passed else "Задание ещё не завершено: исправьте отмеченные строки и повторите проверку.")
    return passed


def check_metric(matrix, length2, basis, displacement):
    if matrix is None or length2 is None:
        print("НЕ ЗАПОЛНЕНО: замените оба None, запустите упражнение и затем проверку.")
        return False
    M, square = numeric(matrix, (2,2)), numeric(length2, ())
    if M is None or square is None:
        print("НУЖНО ИСПРАВИТЬ: M_attempt — матрица 2×2, length2_attempt — одно конечное число.")
        return False
    if not np.allclose(M, basis.T @ basis):
        print("НУЖНО ИСПРАВИТЬ матрицу Грама: её элементы — скалярные произведения столбцов P.")
        return False
    expected = np.linalg.norm(basis @ displacement)**2
    if not np.isclose(square, expected):
        print(f"НУЖНО ИСПРАВИТЬ квадрат длины: получено {float(square):.6f}, геометрическая проверка дала {expected:.6f}.")
        return False
    print(f"УСПЕХ: квадрат длины {float(square):.6f}; длина {np.sqrt(square):.6f}. Матрица Грама согласуется с геометрией.")
    return True

print("Подготовка завершена. Переходите к разделу 1.")

## 1. Робот приехал не туда · 7 минут
Камера прислала пакет `(3, 1)`. Навигация прочитала его как точку карты. Маяк A на карте расположен в `(1, 3)`.

**До запуска графика:** отметьте на бумаге обе точки. Если повернуть камеру, не двигая маяки, какие числа изменятся? Оцените промах.

**Мой прогноз:** …

**Действие:** запустите следующий опыт. Найдите звезду A и крестик остановки на обеих панелях.

**Критерий:** вы можете объяснить, что обе панели показывают одни и те же физические точки, но разными числами.

In [ ]:
scene()
plt.show()

**После наблюдения:** промах — расстояние между остановкой и маяком, а не расстояние от камеры. Согласуется ли число на карточке с вашим прогнозом?

## 2. Как камера описывает точку · 8 минут
Стартуем в начале камеры $a$. Затем делаем $y_1$ шагов вдоль $g_1$ и $y_2$ шагов вдоль $g_2$. Столбцы $P=[g_1\ g_2]$ записаны в координатах карты. Числа $y_1,y_2$ — коэффициенты этих шагов, а не компоненты на осях карты.

**Карточка инструментов:** $x=a+Py$. Матрица перехода «от E к G» построена из нового базиса, но её действие на координаты — **G → E**. Столбцы храним в NumPy одномерными массивами `(2,)`, интерпретируя их как векторы-столбцы.

На диаграмме сопоставьте три стрелки с частями формулы. Найдите координаты маяка A у камеры самостоятельно.

**Ваш код:** замените `None` в `y_attempt` выражением, вычисляющим координаты из `a`, `P`, `x`. Используйте `np.linalg.solve(A, b)`, которая решает систему `A @ z = b`. Оператор `@` означает матричное умножение.

**Критерий успеха:** самопроверка вернёт исходную точку и невязку (длину ошибки) около нуля. Не подбирайте числа по рисунку: выражение должно работать при изменении исходных данных.

In [ ]:
a = np.array([2., 0.])
P = np.array([[0., -1.], [1., 0.]])
x = np.array([1., 3.])
scene(decompose=True)
plt.show()


**Ваш код — координаты маяка:**

In [ ]:
y_attempt = np.linalg.solve(P, x-a)

<details><summary>Подсказка 1</summary>Рассмотрите стрелку от начала камеры к маяку.</details>
<details><summary>Подсказка 2</summary>В координатах карты эта стрелка равна $x-a$.</details>
<details><summary>Подсказка 3</summary>Решите $Py=x-a$: доступна функция np.linalg.solve.</details>

Запустите самопроверку после попытки.

In [ ]:
coordinates_ok = check_coordinates(y_attempt, a, P, x)

## 3. Исправьте программу навигации · 12 минут
Допишите по одной строке в двух функциях. Затем замените ошибочное присваивание `destination_attempt = packet` переводом пакета в систему карты.

Почему `P @ (x-a)` не решает задачу обратного перевода в общем случае? Что нужно знать о матрице, чтобы использовать транспонирование?

**Что изменить:** оба `return None` и строку `destination_attempt`. `to_camera(x, a, P)` принимает точку карты и возвращает её координаты у камеры; `to_map(y, a, P)` выполняет обратный перевод. Используйте аргументы функций, а не числа конкретного примера.

**Критерий успеха:** обе функции проходят три случая, промах вашей остановки равен нулю. Самопроверка рисует маршрут из вашего `destination_attempt`, поэтому исправление кода меняет результат на графике.

In [ ]:
def to_camera(x, a, P):
    return np.linalg.solve(P, x-a)

def to_map(y, a, P):
    return a + P @ y

packet = np.array([3., 1.])
destination_attempt = to_map(packet, a, P)

<details><summary>Подсказка</summary>Проверьте смысл входа и выхода каждой функции. При возвращении на карту сначала составьте линейную комбинацию столбцов, затем добавьте начало камеры.</details>

**Самопроверка:** отдельный блок ниже проверяет также косой базис, чтобы случайное совпадение на повороте не маскировало ошибку.

In [ ]:
navigation_ok = check_navigation(to_camera, to_map, destination_attempt)

## 4. Маршрут тот же, числа другие · 10 минут
Маяки A и B неподвижны на карте. **Предскажите**, изменятся ли их координатная разность и расстояние после поворота камеры на 30° и переноса в `(−1, 1)`.

Изменяйте параметры следующей ячейки и запускайте её заново: это основной интерактивный путь без виджетов. Сравните `corrected=False` и `True`, оба маяка, углы 90° и 30°. Для косой сетки задайте `skew=1`.

При `skew=0` единицы на обеих осях каждой панели имеют одинаковый физический масштаб. При косой сетке правая панель показывает числа в обычной координатной сетке: физические длины читаются по левой панели или через матрицу Грама.

**Порядок опыта:** оставьте `skew=0`; сначала задайте начало `(2, 0)` и угол `90`, затем начало `(-1, 1)` и угол `30`. Запишите разности координат и расстояния в таблицу ниже. Меняйте по одному параметру. `corrected` здесь переключает демонстрационный режим, а не проверяет ваш код раздела 3.

| Начало и угол | Разность у камеры | Расстояние на карте | Норма разности у камеры |
| --- | --- | --- | --- |
| `(2, 0)`, 90° | … | … | … |
| `(-1, 1)`, 30° | … | … | … |

**Критерий:** координаты меняются, расстояние в обоих опытах остаётся 3. В разделе 5 отдельно исследуем, почему для косой сетки этот способ измерения может не работать.

In [ ]:
camera_origin = (-1., 1.)
camera_angle = 30.
selected_beacon = "B"  # "A" или "B"
corrected = True
skew = 0.  # 0: ортонормированный базис; 1: косой
scene(camera_origin, camera_angle, selected_beacon, corrected, skew)
plt.show()

basis = checked_basis(rotation(camera_angle) @ np.array([[1.,skew],[0.,1.]]))
x_A, x_B = np.array([1.,3.]), np.array([4.,3.])
y_A = np.linalg.solve(basis, x_A-np.array(camera_origin))
y_B = np.linalg.solve(basis, x_B-np.array(camera_origin))
print("Разность на карте:", x_B-x_A)
print("Разность у камеры:", np.round(y_B-y_A, 4))
print("Расстояние на карте:", np.linalg.norm(x_B-x_A))
print("Обычная норма разности у камеры:", np.linalg.norm(y_B-y_A))

**Ваш шаг:** раскройте скобки в $x_B-x_A=(a+Py_B)-(a+Py_A)$ и объясните, куда исчезает $a$.

**Моё объяснение:** …

<details><summary>Проверка после объяснения</summary>Одинаковый перенос сокращается: $x_B-x_A=P(y_B-y_A)$. При ортонормированных столбцах $P^TP=I$, поэтому расстояние равно 3 в обеих системах. Для исходной камеры $y_A=(3,1)^T$, $y_B=(3,-2)^T$. При косом базисе обычная норма координатной разности не обязана давать физическую длину.</details>

Смена описания неподвижных маяков — **пассивная**. Поездка робота при фиксированных осях карты — **активное** изменение положения. Для поворота новых осей новые координаты свободного вектора равны $R(\varphi)^Tx_E$; у точки дополнительно вычитается начало.

## 5. Косая сетка обманывает измеритель · ещё 15–25 минут
Пусть $g_1=(1,0)^T$, $g_2=(1,1)^T$ и координаты **перемещения** $y=(1,1)^T$. Перенос начала здесь не участвует.

**Прогноз:** совпадают ли $\|y\|$ и физическая длина $\|Py\|$? Нарисуйте сумму двух шагов до запуска.

In [ ]:
P_skew = np.array([[1.,1.],[0.,1.]])
y_shift = np.array([1.,1.])
show_skew(P_skew, y_shift)
plt.show()

**Карточка инструментов:** $M=P^TP$ — матрица Грама; $\|Py\|^2=y^TMy$. Она хранит длины базисных векторов и их скалярные произведения.

Вычислите `M_attempt` и квадрат физической длины `length2_attempt` через неё.

<details><summary>Подсказка: откуда смешанный член?</summary>$\|y_1g_1+y_2g_2\|^2=y_1^2(g_1\cdot g_1)+2y_1y_2(g_1\cdot g_2)+y_2^2(g_2\cdot g_2)$. В косом базисе средний член обычно не нулевой.</details>

**Ваш код:** используйте `P_skew`, `y_shift` и транспонирование `.T`. В первой строке получите матрицу 2×2, во второй — одно число (квадрат длины, без квадратного корня). Затем запустите отдельную ячейку проверки.

**Критерий:** вычисление через вашу матрицу согласуется с длиной физического перемещения на карте.

In [ ]:
M_attempt = P_skew.T @ P_skew
length2_attempt = y_shift @ M_attempt @ y_shift

In [ ]:
metric_ok = check_metric(M_attempt, length2_attempt, P_skew, y_shift)

**Граница применимости.** Нулевое перемещение имеет длину 0 в любой системе; угол с ним и деление на его норму не определены. Если столбцы совпали, координаты не определяются однозначно. Почти параллельные столбцы могут сильно усиливать малые ошибки.

Предскажите результат для `epsilon=0`, `1e-12`, `1`. Затем запустите ячейку. Порог ниже — защитное вычислительное соглашение, а не определение линейной зависимости.

In [ ]:
epsilon = 1e-12
P_edge = np.array([[1.,1.],[0.,epsilon]])
try:
    checked_basis(P_edge)
    print("Базис прошёл защитную проверку; нулевое перемещение:", np.linalg.solve(P_edge, np.zeros(2)))
except ValueError as error:
    print(error)

### Объясните своими словами
1. Почему пакет `(3,1)` сам по себе не указывает физическую точку без сведений о системе отсчёта?
2. Почему перенос нужен для точки, но сокращается для перемещения?
3. Почему евклидова норма координат верна для повёрнутых единичных осей, но может ошибаться для косых?
4. **Перенос знания:** второй датчик использует начало $b$ и базис $Q$. Как перевести пакет первого датчика непосредственно во второй? Какие матрицы должны быть невырожденными?
5. Исследуйте совпадающие столбцы: может ли одна точка иметь несколько пакетов? Может ли точка вообще не иметь такого описания?

**Мой вывод и способ проверки:** …

**Мост к ИИ.** В реальной системе $x$ могло бы быть положением обнаруженного объекта, а $a,P$ — результатом калибровки датчика. Здесь точки и матрицы заданы вручную. Мы переводим координаты уже найденных точек: не обрабатываем изображение, не моделируем перспективную проекцию и не обучаем модель.

<details><summary>Проверка переноса — откройте после попытки</summary>Сначала $x=a+Py$, затем решите $Qz=a-b+Py$. Для координатных систем нужны невырожденные $P,Q$; непосредственно вычисление $z$ требует решения системы с $Q$.</details>

### Проверка завершения
Запустите после последней проверки каждого задания. Здесь учитываются результаты последних запусков самопроверок; после изменения ответа повторите соответствующую самопроверку.

Для устных заданий проверьте себя: заполнена таблица двух опытов; объяснено сокращение переноса; выполнены три опыта с `epsilon`; написан собственный вывод. Автоматическая проверка не оценивает эти объяснения.

In [ ]:
for label, ok in [("Координаты маяка", coordinates_ok), ("Программа навигации", navigation_ok), ("Матрица Грама (дополнительно)", metric_ok)]:
    print(f"{label}: {'УСПЕХ' if ok else 'ЕЩЁ НЕ ЗАВЕРШЕНО'}")
print("Основной код выполнен." if coordinates_ok and navigation_ok else "Вернитесь к незавершённым заданиям основного пути.")

## Ответы и обсуждение

1. **Прогноз:** при повороте камеры x неподвижного маяка не меняется, y обычно меняется. Ошибочная остановка (3,1), цель (1,3), промах $\sqrt8\approx2.828$. Две панели — одна сцена.
2. **Координаты:** $x-a=(-1,3)^T$, откуда $y=(3,1)^T$. P действует G → E, поэтому для обратного направления решаем систему. $P^T=P^{-1}$ только при ортогональной P. Для данного поворота $P(x-a)=(-3,-1)^T$ — неверный ответ.
3. **Навигация:** исправление — применение `to_map` к пакету. Если поменяли функции, но забыли присваивание, робот продолжает ехать не туда.
4. **Два опыта:** при 90° разность координат B−A равна (0,−3); при 30° — приблизительно (2.598,−1.5). Обе нормы равны 3. Общий перенос сокращается при вычитании точек. Перемещение робота активно; переописание неподвижных маяков пассивно.
5. **Косая сетка:** $M=\begin{pmatrix}1&1\\1&2\end{pmatrix}$, квадрат длины 5, сама длина $\sqrt5$, наивный квадрат 2. Столбцы имеют ненулевое скалярное произведение: смешанный член нельзя отбросить.
6. **Вырождение:** для epsilon=0 столбцы совпадают. Точка со смещением (2,0) имеет бесконечно много описаний, например (2,0), (1,1); смещение (0,1) недостижимо. При 1e−12 базис математически невырожден, но защитная проверка останавливает ненадёжный расчёт. При 1 базис допустим. Нулевая длина допустима; нормирование нулевого вектора — нет.
7. **Другой датчик:** решаем $Qz=a-b+Py$. Невырожденность P и Q нужна для двух полноценных систем координат; непосредственно решение для z требует невырожденности Q.

**Возможный вывод студента:** «Числа зависят от системы. Возвращаю точку на карту и сравниваю с исходной; длину в косом базисе считаю через матрицу Грама». Принимайте эквивалентные объяснения собственными словами.

## Технические проверки эталона
Эти проверки относятся к математической реализации; результаты учебных заданий показаны выше отдельно.

In [ ]:
def reference_to_camera(x, a, P):
    return np.linalg.solve(checked_basis(P), np.asarray(x)-np.asarray(a))

def reference_to_map(y, a, P):
    return np.asarray(a) + checked_basis(P) @ np.asarray(y)

def gram(P):
    P = checked_basis(P)
    return P.T @ P

def physical_distance(delta, P):
    return float(np.sqrt(max(0., np.asarray(delta) @ gram(P) @ np.asarray(delta))))

origin=np.array([2.,0.]); basis=np.array([[0.,-1.],[1.,0.]])
A=np.array([1.,3.]); B=np.array([4.,3.])
u=reference_to_camera(A, origin, basis); v=reference_to_camera(B, origin, basis)
assert np.allclose(u, [3,1]) and np.allclose(v, [3,-2])
assert np.allclose(reference_to_map(u, origin, basis), A)
assert np.allclose(basis @ u, A-origin)
assert np.allclose(basis @ (v-u), [3,0])
assert np.isclose(np.linalg.norm(u-A), np.sqrt(8))
assert np.allclose(gram(basis), np.eye(2))
assert np.isclose(physical_distance(v-u,basis), 3)
S=np.array([[1.,1.],[0.,1.]])
assert np.allclose(gram(S), [[1,1],[1,2]])
assert np.allclose(gram(S), gram(S).T) and np.all(np.linalg.eigvalsh(gram(S))>0)
assert np.isclose(physical_distance([1,1],S)**2, 5)
assert physical_distance([0,0],S)==0
for bad in [np.array([[1.,1.],[0.,0.]]), np.array([[1.,1.],[0.,1e-12]])]:
    try:
        checked_basis(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("Ожидалась защита от ненадёжного базиса")
print("Все эталонные проверки пройдены.")